<a href="https://colab.research.google.com/github/RodrigoGuedesDP/Deeplearning/blob/main/Laboratorio1_TimesNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 1 — Pronóstico en Series de Tiempo Largas (ETT)
## Modelo: **TimesNet** (Wu et al., ICLR 2023)

**Paper:** *TimesNet: Temporal 2D-Variation Modeling for General Time Series Analysis*

**Dataset:** ETT (ETTh1, ETTh2, ETTm1, ETTm2)  
**Tarea:** Predicción de la variable **OT** (Oil Temperature)  
**Entrada:** 96 pasos  
**Horizontes:** {24, 48, 96, 192, 336, 720}  
**Métricas:** MSE, MAE  
**Split:** 70% train, 10% val, 20% test

## 1. Setup e Instalación de Dependencias

In [ ]:
# Instalar dependencias si es necesario
!pip install torch torchvision --quiet
!pip install pandas numpy matplotlib seaborn tqdm --quiet

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Descarga y Carga de Datos

In [ ]:
# Descargar los datasets ETT desde el repositorio oficial
import urllib.request

base_url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/"
files = ["ETTh1.csv", "ETTh2.csv", "ETTm1.csv", "ETTm2.csv"]

os.makedirs("data", exist_ok=True)
for f in files:
    path = f"data/{f}"
    if not os.path.exists(path):
        print(f"Descargando {f}...")
        urllib.request.urlretrieve(base_url + f, path)
    else:
        print(f"{f} ya existe.")
print("\nDescarga completada.")

## 3. Preprocesamiento y Carga de Datos

Siguiendo las instrucciones del laboratorio:
- **Split temporal:** 70% train, 10% validación, 20% test
- **Normalización:** StandardScaler ajustado solo en train
- **Variables de entrada:** HUFL, HULL, MUFL, MULL, LUFL, LULL, OT (7 features)
- **Variable objetivo:** OT

In [ ]:
# --- Funciones de la plantilla del profesor (adaptadas) ---

class StandardScaler:
    """Normalizador estándar: z = (x - mean) / std"""
    def __init__(self, mean, std, eps=1e-6):
        self.mean = mean.astype(np.float32)
        self.std = np.where(std < eps, eps, std).astype(np.float32)
        self.eps = eps

    def transform(self, x):
        return (x - self.mean) / self.std

    def inverse_transform(self, x):
        return x * self.std + self.mean


def read_ETT_csv(path):
    """Lee un archivo CSV del dataset ETT."""
    df = pd.read_csv(path)
    if "date" not in df.columns:
        raise ValueError(f"{path} no tiene columna 'date'.")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True).set_index("date")
    return df


FEATURE_COLS = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
TARGET_COL = "OT"


def split_train_val_test(df, train_ratio=0.7, val_ratio=0.1):
    """Split temporal: 70% train, 10% val, 20% test."""
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return (
        df.iloc[:train_end].copy(),
        df.iloc[train_end:val_end].copy(),
        df.iloc[val_end:].copy(),
    )


def make_scaler(train_df, feature_cols):
    """Crea un scaler basado en los datos de entrenamiento."""
    tr = train_df[feature_cols].dropna()
    mean = tr.mean(axis=0).to_numpy(dtype=np.float32)
    std = tr.std(axis=0, ddof=0).to_numpy(dtype=np.float32)
    return StandardScaler(mean, std)

In [ ]:
class ETTDataset(Dataset):
    """
    Dataset para series de tiempo ETT.
    Genera ventanas deslizantes de (seq_len, n_features) -> (pred_len,) para OT.
    """
    def __init__(self, df, seq_len=96, pred_len=96, target_col="OT",
                 feature_cols=None, scaler=None):
        if feature_cols is None:
            feature_cols = list(df.columns)

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.target_col = target_col
        self.feature_cols = feature_cols
        self.target_idx = feature_cols.index(target_col)

        # Normalizar
        data = df[feature_cols].to_numpy(dtype=np.float32)
        if scaler is not None:
            data = scaler.transform(data)

        self.data = data
        self.n_samples = len(data) - seq_len - pred_len + 1

    def __len__(self):
        return max(0, self.n_samples)

    def __getitem__(self, idx):
        s_begin = idx
        s_end = s_begin + self.seq_len
        r_end = s_end + self.pred_len

        x = self.data[s_begin:s_end, :]                    # (seq_len, n_features)
        y = self.data[s_end:r_end, self.target_idx]         # (pred_len,)

        x = torch.from_numpy(x).float()
        y = torch.from_numpy(y).float()
        return x, y

In [ ]:
# Cargar todos los datasets
datasets_info = {
    "ETTh1": "data/ETTh1.csv",
    "ETTh2": "data/ETTh2.csv",
    "ETTm1": "data/ETTm1.csv",
    "ETTm2": "data/ETTm2.csv",
}

raw_data = {}
for name, path in datasets_info.items():
    raw_data[name] = read_ETT_csv(path)
    print(f"{name}: shape={raw_data[name].shape}, "
          f"rango: {raw_data[name].index.min()} → {raw_data[name].index.max()}")

In [ ]:
# Visualizar las series de tiempo (variable OT)
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, (name, df) in zip(axes.flat, raw_data.items()):
    ax.plot(df.index, df["OT"], linewidth=0.5)
    ax.set_title(f"{name} — Oil Temperature (OT)")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("OT")
plt.tight_layout()
plt.show()

## 4. Implementación de TimesNet

### Idea principal del paper
TimesNet observa que las series de tiempo exhiben **multi-periodicidad**. La idea clave es:

1. **Descubrir periodos** via FFT (Fast Fourier Transform) — seleccionar las top-k frecuencias más significativas.
2. **Transformar la serie 1D a tensores 2D** — reshape basado en cada periodo detectado, de forma que las columnas representen la **variación intra-periodo** y las filas la **variación inter-periodo**.
3. **Procesar con kernels 2D** — usar un bloque Inception (conv2d multi-escala) para capturar patrones en ambas dimensiones.
4. **Agregar adaptativamente** — ponderar las representaciones de distintos periodos usando las amplitudes de la FFT normalizadas con Softmax.

### Arquitectura
- **Embedding:** proyección lineal de C features a d_model dimensiones
- **TimesBlock × N capas** (residual): Period Discovery → Reshape 2D → Inception Block → Reshape back → Adaptive Aggregation
- **Projection head:** capa lineal para generar la predicción del horizonte futuro

### Hiperparámetros del paper (long-term forecasting)
- k = 5 (top frequencies)
- N = 2 layers
- d_model = basado en dim de entrada (min(max(2^ceil(log C), 32), 512))
- lr = 1e-4, batch = 32, epochs = 10
- Loss = MSE

In [ ]:
# ========================
# Inception Block (2D)
# ========================

class InceptionBlockV1(nn.Module):
    """
    Bloque Inception con convoluciones multi-escala (1x1, 3x3, 5x5, 7x7)
    seguido de MaxPool, como en el paper de TimesNet.
    Parámetro eficiente: se comparte para todos los k periodos.
    """
    def __init__(self, in_channels, out_channels, num_kernels=6):
        super().__init__()
        self.num_kernels = num_kernels
        # Múltiples kernels de diferentes tamaños
        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=(k, k), padding=(k//2, k//2))
            for k in range(1, 2 * num_kernels, 2)  # 1, 3, 5, 7, 9, 11
        ])

    def forward(self, x):
        # x: (B, C, H, W)
        out = []
        for conv in self.convs:
            out_i = conv(x)
            # Asegurar mismo tamaño espacial
            if out_i.shape[-2:] != x.shape[-2:]:
                out_i = F.interpolate(out_i, size=x.shape[-2:], mode='bilinear', align_corners=False)
            out.append(out_i)
        # Promedio de todas las escalas
        out = torch.stack(out, dim=-1).mean(-1)
        return out

In [ ]:
# ========================
# TimesBlock
# ========================

def FFT_for_Period(x, k=5):
    """
    Descubre los top-k periodos más significativos via FFT.
    x: (B, T, C)
    Retorna:
        period_list: lista de longitudes de periodo (k,)
        period_weight: amplitudes normalizadas (B, k)
    """
    B, T, C = x.shape
    # FFT
    xf = torch.fft.rfft(x, dim=1)               # (B, T//2+1, C)
    # Amplitud promediada sobre canales
    amplitude = torch.abs(xf).mean(dim=2)         # (B, T//2+1)
    # Ignorar la componente DC (frecuencia 0)
    amplitude[:, 0] = 0
    # Top-k frecuencias
    freq_end = T // 2
    if freq_end < k:
        k = max(1, freq_end)
    _, top_indices = torch.topk(amplitude[:, 1:freq_end+1], k, dim=1)  # (B, k)
    top_indices = top_indices + 1  # compensar el offset por ignorar DC

    # Periodos = T / frecuencia
    # Usar la moda (frecuencia más común en el batch) para cada posición
    top_indices_avg = top_indices.float().mean(dim=0).long()  # (k,)
    period_list = (T / top_indices_avg.float()).long().clamp(min=2)

    # Pesos = amplitud de las frecuencias seleccionadas
    # Gather amplitudes para normalizar
    period_weight = amplitude.gather(1, top_indices)  # (B, k)

    return period_list.tolist(), period_weight


class TimesBlock(nn.Module):
    """
    Un bloque de TimesNet:
    1. FFT para descubrir periodos
    2. Reshape 1D -> 2D para cada periodo
    3. Inception block (2D conv)
    4. Reshape back a 1D
    5. Agregación adaptativa con pesos de amplitud
    """
    def __init__(self, seq_len, pred_len, top_k=5, d_model=64, d_ff=64, num_kernels=6):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.top_k = top_k

        # Inception block compartido
        self.inception = InceptionBlockV1(d_model, d_ff, num_kernels=num_kernels)
        self.conv_out = nn.Conv2d(d_ff, d_model, kernel_size=1)  # proyectar de vuelta

    def forward(self, x):
        """
        x: (B, T, d_model)
        """
        B, T, C = x.shape

        # 1. Descubrir periodos via FFT
        period_list, period_weight = FFT_for_Period(x, self.top_k)

        res_list = []
        for i, p in enumerate(period_list):
            p = int(p)
            if p <= 1:
                p = 2

            # Calcular cuántas filas (periodos completos) y padding necesario
            # rows = ceil(T / p), cols = p
            n_pad = (p - (T % p)) % p
            if n_pad > 0:
                x_padded = F.pad(x, (0, 0, 0, n_pad), mode='constant', value=0)
            else:
                x_padded = x

            T_padded = T + n_pad
            n_periods = T_padded // p

            # 2. Reshape a 2D: (B, T_padded, d_model) -> (B, d_model, n_periods, p)
            x_2d = x_padded.reshape(B, n_periods, p, C).permute(0, 3, 1, 2)
            # x_2d: (B, d_model, n_periods, p) — filas=inter-periodo, cols=intra-periodo

            # 3. Inception 2D
            out_2d = self.inception(x_2d)
            out_2d = self.conv_out(out_2d)  # (B, d_model, n_periods, p)

            # 4. Reshape back a 1D
            out_1d = out_2d.permute(0, 2, 3, 1).reshape(B, T_padded, C)
            # Truncar al largo original
            out_1d = out_1d[:, :T, :]

            res_list.append(out_1d)

        # 5. Agregación adaptativa
        # period_weight: (B, k) -> softmax -> pesos
        period_weight = F.softmax(period_weight, dim=1)  # (B, k)

        # Ponderar y sumar
        out = torch.zeros_like(x)
        for i in range(len(period_list)):
            w = period_weight[:, i].unsqueeze(1).unsqueeze(2)  # (B, 1, 1)
            out = out + w * res_list[i]

        return out

In [ ]:
# ========================
# TimesNet (modelo completo)
# ========================

class TimesNet(nn.Module):
    """
    TimesNet para Long-term Forecasting.

    Arquitectura:
    - Embedding layer (Linear) para proyectar features a d_model
    - N TimesBlocks apilados con conexiones residuales
    - LayerNorm después de cada bloque
    - Projection head: flatten temporal + Linear para generar predicción

    Incluye RevIN (Reversible Instance Normalization) para manejar
    la no-estacionariedad, como se describe en el paper.
    """
    def __init__(self, seq_len=96, pred_len=96, enc_in=7, d_model=64,
                 d_ff=64, n_layers=2, top_k=5, num_kernels=6, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.enc_in = enc_in
        self.d_model = d_model
        self.n_layers = n_layers

        # Embedding: proyección lineal
        self.enc_embedding = nn.Linear(enc_in, d_model)
        self.dropout = nn.Dropout(dropout)

        # TimesBlocks
        self.blocks = nn.ModuleList([
            TimesBlock(
                seq_len=seq_len,
                pred_len=pred_len,
                top_k=top_k,
                d_model=d_model,
                d_ff=d_ff,
                num_kernels=num_kernels
            ) for _ in range(n_layers)
        ])
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_layers)
        ])

        # Temporal projection: para inicializar la predicción futura
        # Usamos MLP temporal como en el paper
        self.temporal_proj = nn.Linear(seq_len, pred_len, bias=True)

        # Output projection: d_model -> 1 (solo OT)
        self.output_proj = nn.Linear(d_model, 1)

    def forward(self, x):
        """
        x: (B, seq_len, enc_in) — serie multivariada normalizada
        returns: (B, pred_len) — predicción de OT
        """
        B, T, C = x.shape

        # --- RevIN: Series Stationarization ---
        # Calcular media y std por instancia
        means = x.mean(dim=1, keepdim=True).detach()  # (B, 1, C)
        x_centered = x - means
        stdev = torch.sqrt(x_centered.var(dim=1, keepdim=True, unbiased=False) + 1e-5).detach()
        x_norm = x_centered / stdev

        # Embedding
        enc_out = self.enc_embedding(x_norm)   # (B, T, d_model)
        enc_out = self.dropout(enc_out)

        # TimesBlocks con residual
        for block, ln in zip(self.blocks, self.layer_norms):
            enc_out = ln(block(enc_out) + enc_out)

        # Temporal projection: (B, T, d_model) -> (B, pred_len, d_model)
        # Transponer para aplicar linear sobre la dim temporal
        enc_out = enc_out.permute(0, 2, 1)      # (B, d_model, T)
        enc_out = self.temporal_proj(enc_out)     # (B, d_model, pred_len)
        enc_out = enc_out.permute(0, 2, 1)       # (B, pred_len, d_model)

        # Output: (B, pred_len, d_model) -> (B, pred_len, 1) -> (B, pred_len)
        out = self.output_proj(enc_out).squeeze(-1)  # (B, pred_len)

        # --- RevIN: De-normalization (solo para el canal OT) ---
        # OT es el último canal (index 6 de 7)
        ot_idx = C - 1  # OT está en la última posición
        out = out * stdev[:, 0, ot_idx:ot_idx+1]
        out = out + means[:, 0, ot_idx:ot_idx+1]

        return out

In [ ]:
# Verificar modelo
model_test = TimesNet(seq_len=96, pred_len=96, enc_in=7, d_model=64, d_ff=64)
x_test = torch.randn(4, 96, 7)
y_test = model_test(x_test)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {y_test.shape}")
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")
del model_test, x_test, y_test

## 5. Entrenamiento y Evaluación

### Configuración (siguiendo el paper)
- **Optimizer:** Adam (lr=1e-4)
- **Scheduler:** Cosine Annealing
- **Loss:** MSE
- **Batch size:** 32
- **Epochs:** 10
- **Early stopping:** basado en validation loss (paciencia 3)

Se entrena un modelo por cada combinación de dataset × horizonte.

In [ ]:
# ========================
# Funciones de entrenamiento
# ========================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    n_batches = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    n_batches = 0
    preds_list = []
    trues_list = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        total_loss += loss.item()
        n_batches += 1
        preds_list.append(pred.cpu())
        trues_list.append(y.cpu())
    avg_loss = total_loss / max(n_batches, 1)
    preds = torch.cat(preds_list, dim=0).numpy()
    trues = torch.cat(trues_list, dim=0).numpy()
    return avg_loss, preds, trues


def compute_metrics(preds, trues):
    """Calcula MSE y MAE."""
    mse = np.mean((preds - trues) ** 2)
    mae = np.mean(np.abs(preds - trues))
    return mse, mae

In [ ]:
def run_experiment(dataset_name, df, pred_len, config, device):
    """
    Entrena y evalúa TimesNet para un dataset y horizonte específico.
    Retorna las métricas de test y el historial de entrenamiento.
    """
    seq_len = config['seq_len']

    # Split
    train_df, val_df, test_df = split_train_val_test(df)
    scaler = make_scaler(train_df, FEATURE_COLS)

    print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # Datasets
    train_ds = ETTDataset(train_df, seq_len=seq_len, pred_len=pred_len,
                          feature_cols=FEATURE_COLS, scaler=scaler)
    val_ds   = ETTDataset(val_df, seq_len=seq_len, pred_len=pred_len,
                          feature_cols=FEATURE_COLS, scaler=scaler)
    test_ds  = ETTDataset(test_df, seq_len=seq_len, pred_len=pred_len,
                          feature_cols=FEATURE_COLS, scaler=scaler)

    print(f"  Ventanas — Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True, drop_last=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=config['batch_size'],
                              shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds, batch_size=config['batch_size'],
                              shuffle=False, num_workers=0)

    # Modelo
    model = TimesNet(
        seq_len=seq_len,
        pred_len=pred_len,
        enc_in=len(FEATURE_COLS),
        d_model=config['d_model'],
        d_ff=config['d_ff'],
        n_layers=config['n_layers'],
        top_k=config['top_k'],
        num_kernels=config['num_kernels'],
        dropout=config['dropout']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizer, T_max=config['epochs'])
    criterion = nn.MSELoss()

    # Training loop con early stopping
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_mse': [], 'val_mae': []}

    for epoch in range(config['epochs']):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_preds, val_trues = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        val_mse, val_mae = compute_metrics(val_preds, val_trues)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mse'].append(val_mse)
        history['val_mae'].append(val_mae)

        lr_now = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:2d}/{config['epochs']} | "
              f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | "
              f"Val MSE: {val_mse:.4f} | Val MAE: {val_mae:.4f} | LR: {lr_now:.2e}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f"  Early stopping en epoch {epoch+1}")
                break

    # Cargar mejor modelo y evaluar en test
    model.load_state_dict(best_state)
    model.to(device)
    test_loss, test_preds, test_trues = evaluate(model, test_loader, criterion, device)
    test_mse, test_mae = compute_metrics(test_preds, test_trues)

    print(f"  *** TEST — MSE: {test_mse:.4f}, MAE: {test_mae:.4f} ***")

    return {
        'dataset': dataset_name,
        'pred_len': pred_len,
        'test_mse': test_mse,
        'test_mae': test_mae,
        'history': history,
        'test_preds': test_preds,
        'test_trues': test_trues,
    }

## 6. Ejecución: Entrenar para todos los datasets y horizontes

Configuración basada en el paper (Table 7):
- `top_k = 5`, `n_layers = 2`, `d_model / d_ff` basado en la dimensión de entrada
- Para ETT con 7 features: d_model = min(max(2^ceil(log2(7)), 32), 512) = 32
- Usamos `d_model = 64` y `d_ff = 64` para mejor rendimiento (dentro del rango del paper)

In [ ]:
# Configuración global
CONFIG = {
    'seq_len': 96,
    'batch_size': 32,
    'lr': 1e-4,
    'epochs': 10,
    'patience': 3,
    'd_model': 64,
    'd_ff': 64,
    'n_layers': 2,
    'top_k': 5,
    'num_kernels': 6,
    'dropout': 0.1,
}

HORIZONS = [24, 48, 96, 192, 336, 720]
DATASET_NAMES = ["ETTh1", "ETTh2", "ETTm1", "ETTm2"]

print("=" * 70)
print("CONFIGURACIÓN DEL EXPERIMENTO")
print("=" * 70)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"  horizontes: {HORIZONS}")
print(f"  datasets: {DATASET_NAMES}")
print("=" * 70)

In [ ]:
# ============================================
# LOOP PRINCIPAL DE ENTRENAMIENTO
# ============================================
all_results = []

for ds_name in DATASET_NAMES:
    for h in HORIZONS:
        print(f"\n{'='*70}")
        print(f"  {ds_name} | input=96, horizon={h}")
        print(f"{'='*70}")

        result = run_experiment(
            dataset_name=ds_name,
            df=raw_data[ds_name],
            pred_len=h,
            config=CONFIG,
            device=device
        )
        all_results.append(result)

        # Liberar memoria GPU
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n\n" + "=" * 70)
print("ENTRENAMIENTO COMPLETADO")
print("=" * 70)

## 7. Resultados

### 7.1 Tabla resumen de métricas (MSE y MAE por horizonte)

In [ ]:
# Tabla de resultados
results_df = pd.DataFrame([
    {
        'Dataset': r['dataset'],
        'Horizon': r['pred_len'],
        'MSE': round(r['test_mse'], 4),
        'MAE': round(r['test_mae'], 4),
    }
    for r in all_results
])

# Pivot table
print("\n" + "=" * 70)
print("RESULTADOS FINALES — MSE y MAE por Dataset y Horizonte")
print("=" * 70)
print()

for metric in ['MSE', 'MAE']:
    pivot = results_df.pivot(index='Dataset', columns='Horizon', values=metric)
    pivot['Promedio'] = pivot.mean(axis=1)
    print(f"\n--- {metric} ---")
    print(pivot.to_string())
    print()

# Tabla completa
print("\n--- Tabla completa ---")
print(results_df.to_string(index=False))

### 7.2 Gráficas de entrenamiento (Loss, MSE, MAE)

In [ ]:
# Gráficas de training loss y validation loss
n_exp = len(all_results)
n_cols = 4
n_rows = math.ceil(n_exp / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flat if n_exp > 1 else [axes]

for i, result in enumerate(all_results):
    ax = axes[i]
    h = result['history']
    epochs_range = range(1, len(h['train_loss']) + 1)
    ax.plot(epochs_range, h['train_loss'], label='Train Loss', marker='o', markersize=3)
    ax.plot(epochs_range, h['val_loss'], label='Val Loss', marker='s', markersize=3)
    ax.set_title(f"{result['dataset']} H={result['pred_len']}", fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Ocultar axes vacíos
for j in range(i+1, len(list(axes))):
    axes[j].set_visible(False)

plt.suptitle("Curvas de Entrenamiento (Train vs Val Loss)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Gráficas de MSE y MAE en validación
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flat if n_exp > 1 else [axes]

for i, result in enumerate(all_results):
    ax = axes[i]
    h = result['history']
    epochs_range = range(1, len(h['val_mse']) + 1)
    ax.plot(epochs_range, h['val_mse'], label='Val MSE', color='tab:blue', marker='o', markersize=3)
    ax2 = ax.twinx()
    ax2.plot(epochs_range, h['val_mae'], label='Val MAE', color='tab:orange', marker='s', markersize=3)
    ax.set_title(f"{result['dataset']} H={result['pred_len']}", fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE', color='tab:blue')
    ax2.set_ylabel('MAE', color='tab:orange')
    ax.legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i+1, len(list(axes))):
    axes[j].set_visible(False)

plt.suptitle("Métricas de Validación (MSE y MAE)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 7.3 Predicciones vs Ground Truth (muestras del test set)

In [ ]:
# Visualizar predicciones vs ground truth para cada dataset (horizonte 96)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, ds_name in zip(axes.flat, DATASET_NAMES):
    # Buscar resultado con horizonte 96
    result = next((r for r in all_results if r['dataset'] == ds_name and r['pred_len'] == 96), None)
    if result is None:
        continue

    preds = result['test_preds']
    trues = result['test_trues']

    # Mostrar las primeras 3 ventanas concatenadas
    n_show = min(3, preds.shape[0])
    pred_concat = preds[:n_show].flatten()
    true_concat = trues[:n_show].flatten()

    ax.plot(true_concat, label='Ground Truth', color='black', linewidth=1)
    ax.plot(pred_concat, label='Predicción (TimesNet)', color='tab:orange', linewidth=1, alpha=0.8)
    ax.set_title(f"{ds_name} — Horizonte 96 (primeras {n_show} ventanas)")
    ax.set_xlabel('Pasos de tiempo')
    ax.set_ylabel('OT (normalizado)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predicciones para diferentes horizontes (un dataset de ejemplo: ETTh1)
horizons_to_show = [24, 96, 336, 720]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, h in zip(axes.flat, horizons_to_show):
    result = next((r for r in all_results if r['dataset'] == 'ETTh1' and r['pred_len'] == h), None)
    if result is None:
        continue

    preds = result['test_preds']
    trues = result['test_trues']

    # Mostrar 1 ventana
    idx = 0
    ax.plot(range(h), trues[idx], label='Ground Truth', color='black', linewidth=1.5)
    ax.plot(range(h), preds[idx], label='Predicción', color='tab:orange', linewidth=1.5, alpha=0.8)
    ax.set_title(f"ETTh1 — Horizonte {h} (MSE={result['test_mse']:.4f})")
    ax.set_xlabel('Pasos de tiempo')
    ax.set_ylabel('OT')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("ETTh1: Predicciones por horizonte", fontsize=14)
plt.tight_layout()
plt.show()

### 7.4 Comparación de métricas por horizonte

In [ ]:
# Bar chart comparativo MSE por dataset y horizonte
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, metric in zip(axes, ['MSE', 'MAE']):
    pivot = results_df.pivot(index='Horizon', columns='Dataset', values=metric)
    pivot.plot(kind='bar', ax=ax, width=0.7)
    ax.set_title(f'{metric} por Horizonte y Dataset')
    ax.set_xlabel('Horizonte de Predicción')
    ax.set_ylabel(metric)
    ax.legend(title='Dataset')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de MSE
pivot_mse = results_df.pivot(index='Dataset', columns='Horizon', values='MSE')
plt.figure(figsize=(10, 4))
sns.heatmap(pivot_mse, annot=True, fmt='.4f', cmap='YlOrRd', linewidths=0.5)
plt.title('MSE por Dataset y Horizonte de Predicción')
plt.ylabel('Dataset')
plt.xlabel('Horizonte')
plt.tight_layout()
plt.show()

# Heatmap de MAE
pivot_mae = results_df.pivot(index='Dataset', columns='Horizon', values='MAE')
plt.figure(figsize=(10, 4))
sns.heatmap(pivot_mae, annot=True, fmt='.4f', cmap='YlOrBr', linewidths=0.5)
plt.title('MAE por Dataset y Horizonte de Predicción')
plt.ylabel('Dataset')
plt.xlabel('Horizonte')
plt.tight_layout()
plt.show()

### 7.5 Comparativa con resultados del paper original

In [ ]:
# Resultados reportados en el paper (Table 2 / Table 13) para TimesNet
# Solo ETT datasets, input=96
paper_results = {
    ('ETTh1', 96): 0.384, ('ETTh1', 192): 0.436, ('ETTh1', 336): 0.491, ('ETTh1', 720): 0.521,
    ('ETTh2', 96): 0.340, ('ETTh2', 192): 0.402, ('ETTh2', 336): 0.452, ('ETTh2', 720): 0.462,
    ('ETTm1', 96): 0.338, ('ETTm1', 192): 0.374, ('ETTm1', 336): 0.410, ('ETTm1', 720): 0.478,
    ('ETTm2', 96): 0.187, ('ETTm2', 192): 0.249, ('ETTm2', 336): 0.321, ('ETTm2', 720): 0.408,
}

# Comparar con nuestros resultados (solo horizontes que coinciden)
comparison = []
for r in all_results:
    key = (r['dataset'], r['pred_len'])
    if key in paper_results:
        comparison.append({
            'Dataset': r['dataset'],
            'Horizon': r['pred_len'],
            'Nuestro MSE': round(r['test_mse'], 4),
            'Paper MSE': paper_results[key],
            'Diff': round(r['test_mse'] - paper_results[key], 4),
        })

if comparison:
    comp_df = pd.DataFrame(comparison)
    print("Comparativa con resultados del paper (MSE):")
    print("(Diff positivo = nuestro resultado es mayor/peor que el paper)")
    print()
    print(comp_df.to_string(index=False))
else:
    print("No hay horizontes coincidentes para comparar.")

## 8. Discusión y Limitaciones

### Observaciones
1. **Tendencia esperada:** El MSE y MAE aumentan con el horizonte de predicción, lo cual es consistente con el paper y la intuición — predecir más lejos en el futuro es más difícil.

2. **Multi-periodicidad:** TimesNet captura eficazmente los patrones periódicos del dataset ETT (periodicidad diaria de 24h en ETTh, y patrones de 96 pasos / 4x por hora en ETTm).

3. **RevIN (Series Stationarization):** La normalización reversible por instancia ayuda significativamente a manejar la no-estacionariedad de las series, como se indica en el paper.

### Limitaciones de nuestra implementación
- **Recursos limitados:** Entrenamos con 10 epochs y early stopping; el paper reporta resultados promediados de 3 corridas.
- **d_model reducido:** Usamos d_model=64 por restricciones de GPU en Colab. El paper usa configuraciones adaptativas que pueden llegar hasta 512.
- **Sin series decomposition:** No implementamos la descomposición seasonal-trend (que el paper muestra no es necesaria para TimesNet, Table 9).
- **Horizontes 24 y 48:** Estos no están en el paper original (que usa {96, 192, 336, 720}), pero son requeridos por el laboratorio.

### Decisiones de diseño
- **Split 70/10/20:** Como indica el laboratorio (el paper usa splits fijos por fechas).
- **Inception block compartido:** Siguiendo el paper, usamos parámetros compartidos entre los k tensores 2D para eficiencia.
- **num_kernels=6:** Kernels de tamaño 1×1 a 11×11 para capturar variaciones multi-escala.


---
## Fin del Laboratorio 1

**Modelo:** TimesNet (Wu et al., ICLR 2023)  
**Implementación:** PyTorch, Google Colab  
**Datasets:** ETTh1, ETTh2, ETTm1, ETTm2  
**Métricas reportadas:** MSE y MAE para horizontes {24, 48, 96, 192, 336, 720}
